In [ ]:
#!pip install ~/fwiVis/utility_functions/

In [ ]:
import fwiVis.fwiVis as fv
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
from math import cos, asin, sqrt
import re

import numpy as np
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
import shapely
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import Point
from shapely import unary_union
import warnings
import folium
import datetime
import time
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
import contextily as cx
from shapely.geometry import box
import sys
from datetime import datetime, timedelta
from itertools import chain

from datetime import date

import seaborn as sn
# sys.path.insert(0, '/projects/old_shared/fire_weather_vis/base-fwi-vis/')
# import fwiVis.fwiVis as fv

## Intro


This is a workflow that follows prost-processing steps that transformed FEDS perimeter assosiated with V2 code (commit #) and traspformed them into the dataset used in the paper. The data needed to be post-processed for a few reasons. 

- Merge bug in V2 of the code


As part of this analysis, we disecered a bug in the V2 code that was not preserving fire-histories when a fire merged into another fire. That bug was fixed in FEDS v3, and a post-hoc correctino was applied ot the data in the paper. 


- Fixing assumptions about how long a fire can be inactive before it is considered "ended"

Developed in California, FEDS V2 made the assumption that a fire that had no new fire pixels for 5 days was "inactive". In the Quebec 2023 fire season, there were several fires that violated this assumption, and would have boarders flare up after periods of inactivity. This created a false inflation of spread, because the algorithm would register two fires superimposed along the same boarders. To correct for this, we merged fires that had spatial intersection. This is a conservative assumption. 



# Initial repocessing of the merge error


This is the workflow for reprocessing the merge bug found in the V2 code. 


In [2]:
def unique(list1):
    ans = reduce(lambda re, x: re+[x] if x not in re else re, list1, [])
    return(ans)

def remerge_largefire(fires):
    '''
    If two final feds perimeters intersect spatialy, check if one ended before the other began. If yes, give it the ID of the earlier perimeter. 
    Note: Could optionally be spitting out time differences, or sorting by them.
    '''
    # Get an id, first/ last t per ID, and a geometry
    first_perims = fires[~fires.geometry.isnull()].groupby("fireID").t.min().reset_index()
    last_perims = fires[~fires.geometry.isnull()].groupby("fireID").t.max().reset_index()
    plot_last = fires.merge(last_perims, on = ["fireID", "t"], how = 'right')
    plot_first = fires.merge(first_perims, on = ["fireID", "t"], how = 'right')
    plot_last = plot_last[["fireID", "t", "geometry"]]
    plot_first = plot_first[["fireID", "t", "geometry"]]
                    
    # Check what perimeters spatially intersect into anouther one through time
    last_in_last = plot_last.sjoin(plot_last, how = "left", predicate = "intersects")
    lil = last_in_last.groupby(["fireID_left",]).t_right.min().reset_index()
    id_with_max_time = lil.merge(last_in_last[["t_right", "t_left", "geometry", "fireID_right", "fireID_left"]], on = ["t_right", "fireID_left"], how = "right")

    id_with_max_time = id_with_max_time.rename(columns={"fireID_right": "mergeID", 
                                     "fireID_left" :"fireID",
                                     "t_right":"mergeID_t",
                                     "t_left":"fireID_end_t"
    })

    id_with_max_time_check = id_with_max_time[id_with_max_time.fireID_end_t < id_with_max_time.mergeID_t]

    ### FireID the earlier perimeter that later perimeters are merged into. "mergeID" describes the merge-ey ## I actually think this is wrong. I think I made it so the "fireID" is what is merged into the "mergeID"
    fireID_with_merge = id_with_max_time_check.groupby(["fireID"]).mergeID.unique().reset_index() 
 

    mergeID_with_fireID  = id_with_max_time_check.groupby(["mergeID"]).fireID.unique().reset_index()
                    
    # Check when the fireID and mergeID started/stopped. 
    get_merge_start = plot_first[["fireID", "t"]].rename(columns={"fireID":"mergeID", 
                                                           "t":"mergeID_start_t"})

    get_fireID_start =  plot_first[["fireID", "t"]].rename(columns={ "t":"fireID_start_t"})
 

    id_map = id_with_max_time_check.merge(get_merge_start, on = ["mergeID"])
    id_map = id_map.merge(get_fireID_start, on = ["fireID"])
    id_map = id_map[["fireID","fireID_start_t",  "fireID_end_t", "mergeID", "mergeID_start_t", "mergeID_t"]]
    id_map["time_diff_fireIDend_mergeIDstart"] = id_map.fireID_end_t.astype('datetime64[ns]') - id_map.mergeID_start_t.astype('datetime64[ns]') ## Negative means that mergeID started after fireID ended
    return(id_map)

def merge_fires_into(newfire, fires):
    '''
    newfire [DataFrame] output from remerge_largefire. 
    fires [GeoDataFrame] geodataframe of all largefires. 
    '''
    fires = fires[~fires.geometry.isna()]
    print("Removing full timeseries of met data.")
    remake_fires = fires.copy() ## Need to do this? 
    remake_fires["composit_ids"] = ""
    remake_fires["rows_edited"] = False ## Will be none if fire doesn'ge merge into a different fire

    fires.t = fires.t.astype("datetime64[ns]")
    separator = ', '
    for m in newfire.mergeID.unique(): ### For each id that had something merged into it
        fireIDs = newfire[newfire.mergeID == m].fireID.unique() ## Get all the fireID of things that were merged into it
        row_mask = (fires.fireID.isin(fireIDs) | fires.fireID.isin([m]) ) ## Get the IDs that will merge into m and m
        times = fires[row_mask].t.unique()

        for t in times:
            #print(t)
            row_mask_f = (row_mask) & (fires.t.astype("str") == str(t))
            row_mask_t = (remake_fires.fireID == m) & (remake_fires.t.astype("str") == str(t))
            row_mask_any_t =  (remake_fires.fireID.isin(fireIDs) | remake_fires.fireID.isin([m]) ) & (remake_fires.t.astype("str") == str(t))
            if row_mask_f.any():  # Check if any rows match row_mask_t
                remake_fires.loc[row_mask_any_t, "geometry"] = unary_union(fires[row_mask_f].geometry)
                remake_fires.loc[row_mask_any_t, "farea"] = fires.loc[row_mask_f, "farea"].sum()
                remake_fires.loc[row_mask_any_t, "fperim"] = fires.loc[row_mask_f, "fperim"].sum()
                remake_fires.loc[row_mask_any_t, "n_pixels"] = fires.loc[row_mask_f, "n_pixels"].sum()
                remake_fires.loc[row_mask_any_t, "n_newpixels"] = fires.loc[row_mask_f, "n_newpixels"].sum()
                remake_fires.loc[row_mask_any_t, "flinelen"] = fires.loc[row_mask_f, "flinelen"].sum()
                remake_fires.loc[row_mask_any_t, "duration"] = fires.loc[row_mask_f, "duration"].max()
                remake_fires.loc[row_mask_any_t, "pixden"] = fires.loc[row_mask_f, "n_newpixels"].sum() / fires.loc[row_mask_f, "farea"].sum()
                remake_fires.loc[row_mask_any_t, "meanFRP"] = (fires.loc[row_mask_f, "meanFRP"] * fires.loc[row_mask_f, "farea"]).sum() / fires.loc[row_mask_f, "farea"].sum()
                remake_fires.loc[row_mask_any_t, "composit_ids"] = separator.join(fires.loc[row_mask_f, "fireID"].unique())
                remake_fires.loc[row_mask_any_t, "rows_edited"] = "edited"
                remake_fires.loc[row_mask_any_t, "fireID"] = m
            else:

                remake_fires.loc[row_mask_any_t, "rows_edited"] = f"No row of {m} at {t}"
    
     
    remake_fires = remake_fires[~remake_fires.fireID.isin(newfire.fireID)] ## Drop the IDs that will only be merged into something else
    
    return(remake_fires)

In [3]:
path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Final_dataset_as_of_20240209.csv" ## The raw dataset generated by V2 Feds
fires = fv.prep_fire_files(path)
fires = fires[fires.InterCloud.isna()]
fires.t = pd.to_datetime(fires.t, format='ISO8601')
new_fires = remerge_largefire(fires)
test =  merge_fires_into(new_fires, fires)
#test.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_all_fires_merged_together_06172024.csv") # This is the "remerged" dataset I use in the paper. 
test.to_csv(f"/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_all_fires_merged_together_{datetime.today().strftime('%Y%m%d')}.csv")

Removing full timeseries of met data.


/projects/myenvs/fireatlas_oct4/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/tmp/ipykernel_6036/1394322342.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'edited' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  remake_fires.loc[row_mask_any_t, "rows_edited"] = "edited"


# Reprocessing To Correct Fire Inactivity Assuptions

In [4]:
def get_centroid(df):
    df_tmp = df[~df.geometry.isnull()]
    df_tmp["farea_unitless"] = df_tmp.geometry.area
    max_u = df_tmp.farea_unitless.max()
    lat = df_tmp[df_tmp.farea_unitless == max_u].centroid.y
    lon = df_tmp[df_tmp.farea_unitless == max_u].centroid.x

    df["lat_centroid"] = lat.iloc[0]
    df["lon_centroid"] = lon.iloc[0]
    return(df)

def chop_fires_at_end(df):
    final_t = df[df.n_newpixels > 0].t.max()
    df = df[df.t <= final_t]
    return(df)

def diff_fix(df, col = "farea"):
    #df = df2
    fid = "fire"
    df = df.sort_values(by = "t")
    df[col+ "_diff"] = df[col].diff()

    df[col+ "_diff_round"] = df[col+ "_diff"].round(4) ## Correcting for negatives introduced by numarical instabilities/ rounding differences and create very small negative numbers
    
    ### Go through and finding times when fire area is negative, checking for change in area from previous, and using that number as "merge ignition". 
    df["geometry_difference_manual"] = None
    df["area_from_geometry_difference_manual"] = None
    fireID_t_pairs = df.loc[(df.farea_diff_round < 0), ["t"]].reset_index(drop = True)
    for i in range(0, len(fireID_t_pairs)):
        t = str(fireID_t_pairs.iloc[i].iloc[0])
        t_obj = datetime.strptime(t, "%Y-%m-%d %H:%M:%S")
        t_previous = t_obj - timedelta(hours=12)
        t_previous = t_previous.strftime("%Y-%m-%d %H:%M:%S")
        row_mask_t =  (df.t == t) # (df.fireID == fid) &
        row_mask_t_prev =  (df.t == t_previous) # (df.fireID == fid) &
        if( len(df.loc[row_mask_t_prev, "geometry"]) == 0 ):
            print(f"Warning: {fid} does not have t {t_previous}. Trying timestep 12 hours earlier.")
            t_previous = t_obj - timedelta(hours=24)
            t_previous = t_previous.strftime("%Y-%m-%d %H:%M:%S")
            row_mask_t_prev = (df.t == t_previous) # (df.fireID == fid) & 

            if(len(df.loc[row_mask_t_prev, "geometry"]) == 0 ):
                print(f"Warning: {fid} STILL does not have t {t_previous}. Attempting to extract next best timestep.")
                t_previous = df.loc[ (df.t < t)].t.max() # (df.fireID == fid) &
                row_mask_t_prev = (df.t == t_previous) # (df.fireID == fid) & 
                print(f"Warning: previous timesetp being set to {t_previous}.")

        df.loc[row_mask_t, ["geometry_difference_manual"]] = df.loc[row_mask_t, "geometry"].drop_duplicates().iloc[0].symmetric_difference(df.loc[row_mask_t_prev, "geometry"].drop_duplicates().iloc[0])
        df.loc[row_mask_t, "area_from_geometry_difference_manual"] =  round(df.loc[row_mask_t, ["geometry_difference_manual"]].drop_duplicates().iloc[0].iloc[0].area, 4) ## Round to correct weird numerical stuff
    
    conditional = df[col+ "_diff"].isna()
    
    
    if(len(df.loc[conditional, col+ "_diff"]) == 1):

        df.loc[conditional, col+ "_diff"] = df.loc[conditional, col] ## Setting the first value column that the differenceing is happening in. In fire area, represents the ingition area
    else:
        print(f"ERROR: More that one nan value in diff! Does not represent first value of {col}!" )
        
        
    df.loc[df[col+ "_diff_round"] < 0, [col+ "_diff"]] = df.loc[df[col+ "_diff_round"] < 0 , ["area_from_geometry_difference_manual"]]
    return(df)

def average_var_from_multiple_fires(remerged):

    FWI_merge = remerged.groupby(["fireID", "t"]).FWI.mean().reset_index()
    FWI_merge = FWI_merge.dropna()
    FWI_merge


    cp_remerge = remerged

    cp_remerge = remerged[['fireID', 't', 'geometry', 'n_pixels', 'n_newpixels', 'farea', 'fperim',
           'flinelen', 'duration', 'pixden', 'meanFRP', 'composit_ids',
           'rows_edited']]

    cp_remerge = cp_remerge.merge(FWI_merge, on = ["fireID", "t"], how = "inner")
    cp_remerge = cp_remerge.drop_duplicates()
    return(cp_remerge)



In [5]:
def get_ids(df, var = "farea", sep = ","):
    
    ids = sep.join(df.fireID.unique())
    return(ids)

def test_if_ids_are_equal(df, id_var = "fireID", indicator_var = "farea"):
    group = df.groupby("fireID").farea.max().reset_index()
    ids = group.groupby("farea").apply(get_ids).reset_index()
    ids = ids.rename(columns={0:"match_ids"})

    ids[ids.match_ids.str.contains(",")]

    for m in ids[ids.match_ids.str.contains(",")].match_ids:
        all_ids = m.split(",")
        num_ids = len(all_ids)
        ## Check if the first ID is equal to last. I dont' want to go through an check each one. 
        tmp = df[df.fireID == all_ids[0]].drop(columns = "fireID", axis = 1).reset_index(drop = True)
        for l in range(1, num_ids):
            bool = tmp.equals(df[df.fireID == all_ids[l]].drop(columns = "fireID", axis = 1).reset_index(drop = True))
            print(bool)
            if (not bool):
                print(f" ids {all_ids[0]} and {all_ids[l]} are not equal!")
                return(None)
            else:
                return(df)

def fix_chaining_fires(ids, fires, fires_unmerged):
    '''
    A method that will look for fires that have intersecting area. This is a tack-on method when fires area areas that don't merge sensibly: ie will touch but not merge, or will overlap partially in time, or will overlap with one fire, which overlaps with another fire but not another. 
    fires_unmerged (GeoDataFrame). The FEDS Data pre-merge. /projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Final_dataset_as_of_20240209.csv
    fires (Geodataframe). Fires with preliminary merging done. Data generated from remerge_make_fires_with_full_history-Copy2.ipynb. saved id /projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_all_fires_merged_together_06172024.csv. 
    
    ids: a list of ID that have been idetified as having overlapping boundaries. (See script below for modified remerge_largefire that is based on maximum area not maximum time). 
    '''
    fires2 = fires.copy()
    fires2 = fires2[['fireID', 't', 'geometry', 'n_pixels', 'n_newpixels',
       'farea', 'fperim', 'flinelen', 'duration', 'pixden', 'meanFRP',
       'csv_geometry', 'lon_centroid', 'lat_centroid', 'GEOS-5.IMERGEARLY',
       'FWI', 'FWI_lead_1', 'FWI_lead_2', 'FWI_lead_3', 'FWI_lead_4',
       'FWI_lead_5', 'FWI_lead_6', 'FWI_lead_7', 'FWI_lead_8', 'pre_fire',
       'composit_ids', 'rows_edited']]
    df_list = []
    separator = ', '
    ### Get the bounding box of the largest shape in the fire
    for i in ids:

        buffer = fires.loc[(fires.fireID == i) & (fires.unitless_area == fires[fires.fireID == i].unitless_area.max()), "geometry"].buffer(0.05).iloc[0]
        unique_ids = fires_unmerged[fires_unmerged.intersects(buffer)].fireID.unique()


        foo = fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids)]

        t_of_ids = fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids)].t.unique()
         
        for t in t_of_ids:
            ### Make a smaller df of fires @ different times from fire_unmerged. Can't do it from fires becuase missing some "t" values
            foo.loc[(foo.t == t), "geometry"] = unary_union(fires_unmerged[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t)].geometry)
            foo.loc[(foo.t == t), "fireID"] = i
            foo.loc[(foo.t == t), "composit_ids"] = separator.join(fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "fireID"].unique())
            foo.loc[(foo.t == t), "rows_edited"] = "edited"
            foo.loc[(foo.t == t), "farea"] = fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "farea"].sum()
            foo.loc[(foo.t == t), "fperim"] = fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "fperim"].sum()
            foo.loc[(foo.t == t), "n_pixels"] = fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "n_pixels"].sum()
            foo.loc[(foo.t == t), "n_newpixels"] = fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "n_newpixels"].sum()
            foo.loc[(foo.t == t), "flinelen"] = fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "flinelen"].sum()
            foo.loc[(foo.t == t), "duration"] = fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "duration"].max()
            foo.loc[(foo.t == t), "pixden"] = fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "n_newpixels"].sum() / fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "farea"].sum()
            foo.loc[(foo.t == t), "meanFRP"] = (fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "meanFRP"] * fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "farea"]).sum() / fires_unmerged.loc[fires_unmerged.fireID.isin(unique_ids) & (fires_unmerged.t == t), "farea"].sum()
            foo = foo[['fireID', 't', 'geometry', 'n_pixels', 'n_newpixels',
       'farea', 'fperim', 'flinelen', 'duration', 'pixden', 'meanFRP',
       'csv_geometry', 'lon_centroid', 'lat_centroid', 'GEOS-5.IMERGEARLY',
       'FWI', 'FWI_lead_1', 'FWI_lead_2', 'FWI_lead_3', 'FWI_lead_4',
       'FWI_lead_5', 'FWI_lead_6', 'FWI_lead_7', 'FWI_lead_8', 'pre_fire',
       'composit_ids', 'rows_edited']]
        ### Delete incorrect entries in fires
        fires2 = fires2[fires2.fireID != i]
        df_list.append(foo)



    ### add df to fires df
    df_list.append(fires2)
    #df_list = [df.reset_index(drop=True) for df in df_list]
    #return(df_list)
    corrected_merged_fires = pd.concat(df_list, ignore_index=True)
    
    group = corrected_merged_fires.groupby("fireID").farea.max().reset_index()
    ids = group.groupby("farea").apply(get_ids).reset_index()
    ids = ids.rename(columns={0:"match_ids"})
    
    test_if_ids_are_equal(corrected_merged_fires)
    
    #merge_ids = ids.match_ids.str.split(',').str[0].str.strip()
    redundant_ids = [*ids.match_ids.apply(lambda x: [item.strip() for item in x.split(',')[1:]]).explode().dropna()]
    #print(corrected_merged_fires.composit_ids)
    corrected_merged_fires = corrected_merged_fires[(~corrected_merged_fires.fireID.isin(redundant_ids))]
    
    
    return(corrected_merged_fires)

 Read in both the merged and unmerged files, and check for fires that have spatial intersection.

In [6]:
path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Final_dataset_as_of_20240209.csv"  ## The CSV I used in the paper
fires_unmerged = fv.prep_fire_files(path)
fires_unmerged = fires_unmerged.to_crs("EPSG:4326")
fires_unmerged = fires_unmerged[~fires_unmerged.geometry.isnull()]
fires_unmerged = fires_unmerged[fires_unmerged.InterCloud.isna()]
fires_unmerged.t = pd.to_datetime(fires_unmerged.t, format='ISO8601')


path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_all_fires_merged_together_06172024.csv" # Fires with preliminary merging
fires = fv.prep_fire_files(path) ## Check crs
fires = fires[~fires.geometry.isna()]
fires = fires.sort_values(by = ["fireID", "t"])
fires.t = pd.to_datetime(fires.t, format='ISO8601')
fires = fires.to_crs("EPSG:4326")

fires["unitless_area"] = fires.geometry.area


first_perims = fires[~fires.geometry.isnull()].groupby("fireID").t.min().reset_index()
last_perims = fires[~fires.geometry.isnull()].groupby("fireID").t.max().reset_index()
plot_last = fires.merge(last_perims, on = ["fireID", "t"], how = 'right')

## Better than last perim because with merge fixes the last perim is not the largest
area_max = fires[~fires.geometry.isnull()].groupby("fireID").unitless_area.max().reset_index()
plot_area = fires.merge(area_max, on = ["fireID", "unitless_area"], how = 'right')

plot_area = plot_area[~plot_area.geometry.isnull()].groupby("fireID").t.max().reset_index()
plot_area_merge = fires.merge(plot_area, on = ["fireID", "t"], how = 'right')
plot_first = fires.merge(first_perims, on = ["fireID", "t"], how = 'right')
# plot_last = plot_last[["fireID", "t", "geometry"]]
plot_first = plot_first[["fireID", "t", "geometry"]]

max_area_int_with_max_area = plot_area_merge.sjoin(plot_area_merge, how = "left", predicate = "intersects")  ### Kills the kernel
max_area_int_with_max_area = max_area_int_with_max_area[max_area_int_with_max_area.fireID_left != max_area_int_with_max_area.fireID_right]
max_area_int_with_max_area = max_area_int_with_max_area.drop_duplicates()


/tmp/ipykernel_6036/3412472914.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  fires["unitless_area"] = fires.geometry.area


In [7]:
remerged = fix_chaining_fires(max_area_int_with_max_area.fireID_left.unique(), fires, fires_unmerged)
remerged = remerged.drop_duplicates()
#remerged.to_csv(f"projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_all_fires_merged_together_plus_area_fix_09042024.csv") # The file used in the paper. 
remerged.to_csv(f"/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Quebec_all_fires_merged_together_plus_area_fix_{datetime.today().strftime('%Y%m%d')}.csv")

/tmp/ipykernel_6036/3993848109.py:47: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  buffer = fires.loc[(fires.fireID == i) & (fires.unitless_area == fires[fires.fireID == i].unitless_area.max()), "geometry"].buffer(0.05).iloc[0]
/projects/myenvs/fireatlas_oct4/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/projects/myenvs/fireatlas_oct4/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_i

True
